### Data preprocessing

This notebook prepares monthly mean anomalies from observations and the TOMCAT CTM simulation that are further used for the causal inference. 

In [ ]:
# Imports
import numpy as np
%matplotlib inline     
import os
import xarray as xr
import pandas as pd
import glob
#ignore warnings
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.append('/path/to/the/helper_functions/')
from helper_functions import extract_year_mls, monthly_anomalies, process_dataset

# =========================================================
#                         SET-UP 
# =========================================================

start, end = "2004-01", "2021-12"
time_info = pd.date_range(start=start, end=end, freq='MS')
#Define lat and vertical grid to ensure that all datasets are consistent
lat_grid = np.arange (-60,70,10)
lvl_grid = np.array([
    500.0, 450.0, 400.0, 350.0, 300.0, 285.0, 250.0, 200.0, 170.0, 150.0,
    130.0, 115.0, 100.0, 90.0, 80.0, 70.0, 60.0, 50.0, 40.0, 35.0,
    30.0, 25.0, 20.0, 15.0, 10.0, 7.0, 5.0])
# list of observations to be used
obs_data_list = ['osiris', 'mls', 'era5']
# list vars to be analyzed from ERA5
if 'era5' in obs_data_list: 
    vars_era5 = ['wtem', 'ta']

#### Down below provide the paths to original data

In [ ]:
paths = {
    "osiris": "/path/to/osiris/data/v7.3/",
    "mls": "/path/to/MLS_Aura/data/v5/",
    "era5_w": "/path/to/era5/wtem/data/",
    "era5_ta": "/path/to/era5/ta/data/",
}

### Load and preprocess observations (satellite data and reanalysis)

In [ ]:
data_arrays = []
   
for obs_data in obs_data_list: 
    if obs_data == 'osiris': 
        data_path = paths["osiris"]
        # List here the vars that should be preprocessed from the OSIRIS files
        obs_var_list= ['ozone', 'no2']
        for obs_var in obs_var_list: 
            if obs_var == "ozone": 
                data =  xr.open_dataset(data_path + 'osiris_v73_ozone_MZM_vmr_plevs_5deg_shift.nc')
            elif obs_var == "no2" or obs_var == "no" or obs_var == "nox": 
                data   =  xr.open_dataset(data_path + 'osiris_v73_nox_MZM_vmr_plevs_5deg_shift.nc')  
                
            # Slice for selected period and interpolate data to the common grid
            data = data.sel(time=slice(start, end)).interp(latitude=lat_grid).sel(pressure=slice(lvl_grid[0], lvl_grid[-1])).transpose("time", "pressure", "latitude")
            
            # Below OSIRIS data needs reindexing because of missing Junes for 2018-2021 due to eclipse. 
            # Nans are inserted for these dates
            if obs_var == "ozone": 
                data_array = xr.DataArray(data.ozone_vmr.values, dims=["time", "lvl", "lat"], 
                              coords={"time": data["time"], "lvl": lvl_grid, "lat": lat_grid,}, 
                              name='osiris_o3').reindex(time=time_info)  
            elif obs_var == "no2": 
                data_array = xr.DataArray(data.no2_concentration.values, dims=["time", "lvl", "lat"], 
                              coords={"time": data["time"], "lvl": lvl_grid, "lat": lat_grid,}, 
                              name='osiris_no2').reindex(time=time_info)  

            elif obs_var == "no": 
                data_array = xr.DataArray(data.no_concentration.values, dims=["time", "lvl", "lat"], 
                              coords={"time": data["time"], "lvl": lvl_grid, "lat": lat_grid,}, 
                              name='osiris_no').reindex(time=time_info)  

            elif obs_var == "nox": 
                data_array = xr.DataArray(data.nox_concentration.values, dims=["time", "lvl", "lat"], 
                              coords={"time": data["time"], "lvl": lvl_grid, "lat": lat_grid,}, 
                              name='osiris_nox').reindex(time=time_info)  
            data_arrays.append (data_array)
            
    elif obs_data == 'mls': 
        ds_FILES = paths["mls"]
        # Use glob to find all .nc files in the folder
        nc_files = glob.glob(os.path.join(ds_FILES, '*.nc'))
        # Sort the files by the extracted year
        nc_files.sort(key=extract_year_mls)
        # Load MLS data
        data = xr.open_mfdataset(nc_files, combine='nested', concat_dim='time', group='N2O PressureZM', engine='netcdf4')
        data = data.sel(time = slice(start, end)).interp(lat=lat_grid, lev=lvl_grid)
        data_array = xr.DataArray(data.value.values, dims=["time", "lvl", "lat"], 
                              coords={"time": time_info, "lvl": lvl_grid, "lat": lat_grid,}, 
                              name='mls_n2o')
        data_arrays.append (data_array)
    elif obs_data == 'era5':
        for var_era5 in vars_era5:
            if var_era5 == 'wtem':
                ds_FILES = paths["era5_w"]
                # Use glob to find all .nc files in the folder
                nc_files = glob.glob(os.path.join(ds_FILES, '*.nc'))
                data = xr.open_mfdataset(nc_files, engine='netcdf4')
                data = data.sel(time=slice(start, end))
                data = data.sel(lon=0).interp(lat=lat_grid).assign_coords(plev=data['plev']/100).isel(plev=slice(None, None, -1)).interp(plev=lvl_grid)
                data_array = xr.DataArray(data.wtem.values, dims=["time", "lvl", "lat"], 
                                    coords={"time": time_info, "lvl": lvl_grid, "lat": lat_grid,}, 
                                    name='era5_w')
                data_arrays.append (data_array)
            else:
                ds_FILES = paths["era5_ta"]
                nc_files = glob.glob(os.path.join(ds_FILES, '*.nc'))
                nc_files = [f for f in nc_files if start[:4] <= f.split('_')[-2] <= end[:4]]
                data = xr.open_mfdataset(nc_files, engine='netcdf4')
                data = data.mean(dim='longitude').sortby('latitude').interp(latitude=lat_grid).interp(level=lvl_grid) #.sortby to reverse the order
                data_array = xr.DataArray(data.t.values, dims=["time", "lvl", "lat"], 
                                    coords={"time": time_info, "lvl": lvl_grid, "lat": lat_grid,}, 
                                    name='era5_ta')
                data_arrays.append (data_array)
        data_arrays.append (data_array)
ds = xr.merge(data_arrays)
#Calculate monthly anomalies for selected variables
ds_anomalies = xr.Dataset({var: monthly_anomalies(ds[var]) for var in ds.data_vars})
ds_anomalies = ds_anomalies.rename({var: f"{var}_anomaly" for var in ds_anomalies.data_vars})
# Merge monthly timeseries and their anomalies into one file
ds = xr.merge([ds, ds_anomalies])
#save calculated data into the .nc file
ds.to_netcdf(f'/path/to/save/the/file/obs_data_collection_interpolated_{start[:4]}-{end[:4]}.nc')

### Load and preprocess the TOMCAT CTM simulation

In [ ]:
ds_FILES_chemistry = '/path/to/the/TOMCAT/CTM/simulation/'
#list of variables as they are defined in TOMCAT files
vars = ['O3_mm', 'NO2_mm', 'N2O_mm', 'WSTAR', 'uwind_mm', 'te_mm']
time_, lat_ = (len (time_info), len (lat_grid)) 
var_fin = {var: np.zeros((time_, len(lvl_grid), lat_)) for var in vars}
# Loop through variables, time, and latitudes

for var in vars:
    print (var, 'is processed....')
    if var == "WSTAR":
        data = (
            xr.open_dataset('/path/to/TOMCAT/wstar/dataset.nc')
            .sel(longitude = 0).interp(latitude=lat_grid)
            .sel(time = slice(start, end)).resample(time='1M').mean()
            .isel(level=slice(None, None, -1)).interp(level=lvl_grid) 
            ) 
        var_fin[var] = data.WSTAR.data
        
    else: 
        data = xr.open_dataset(ds_FILES_chemistry+ 'TOMCAT_CTM_Simulation_2004-2021mm.nc')               
        data = (data
            .isel(lat=slice(None, None, -1))  # Reverse latitude order [from 90...-90] to [-90...90]
            .interp(lat=lat_grid)  # Interpolate to new latitude grid (identical to obs)
            .sel(time=slice(start, end))
            .mean(dim="lon")  # Average over longitude
        )
        #from Pa to hPa
        pressure = data.p_mm / 100  
        #use monthly mean pressure to interpolate var to a common pressure grid
        for idx in range(time_):
            for idz in range(lat_):
                var_fin[var][idx, :, idz] = np.interp(lvl_grid, pressure[idx, :, idz], data[var].values[idx, :, idz])

# Save data from dict to dataset 
print ('save data to dataset')
ds = xr.Dataset(
    {var: (['time', 'lvl', 'lat'], var_fin[var]) for var in vars},
    coords={
        'time': time_info,  
        'lvl': lvl_grid,           
        'lat': lat_grid,  
    }
)


print ('calculate anomalies')
ds_anomalies = xr.Dataset({var: monthly_anomalies(ds[var]) for var in vars})
ds_anomalies = ds_anomalies.rename({var: f"{var}_anomaly" for var in ds_anomalies.data_vars})
ds = xr.merge([ds, ds_anomalies])
ds.to_netcdf(f'/path/to/save/the/file/ctm741_data_collection_interpolated_{start[:4]}-{end[:4]}.nc')


O3_mm is processed....
NO2_mm is processed....
N2O_mm is processed....
WSTAR is processed....
uwind_mm is processed....
te_mm is processed....
save data to dataset
calculate anomalies


### Check the final .nc files (optional).


In [ ]:

data_sources = ["OBS" , "TOMCAT"]
# variables to be analyzed
var_names = ["w*", "N$_2$O", "NO$_2$", "O$_3$", "T"]

base_in = "/work/bd0854/b380971/ozone_causal_discovery_ACP_rev/preprocessed_data/"
files = {
    "TOMCAT": f"ctm741_data_collection_interpolated_{start[:4]}-{end[:4]}.nc",
    "OBS": f"obs_data_collection_interpolated_{start[:4]}-{end[:4]}.nc",
}

variable_mappings = {
    "TOMCAT": {
        "NO2_mm_anomaly": (r"NO$_2$", 1e9),
        "O3_mm_anomaly": (r"O$_3$", 1e6),
        "N2O_mm_anomaly": (r"N$_2$O", 1e9),
        "WSTAR_anomaly": (r"w*", 1000),
        "te_mm_anomaly": (r"T", 1),
    },
    "OBS": {
        "osiris_no2_anomaly": (r"NO$_2$", 1e9),
        "osiris_o3_anomaly": (r"O$_3$", 1e6),
        "mls_n2o_anomaly": (r"N$_2$O", 1e9),
        "era5_w_anomaly": (r"w*", 1000),
        "era5_ta_anomaly": (r"T", 1),
    },
}


df_dict = {}

for source in data_sources:
    df = process_dataset(base_in, files[source], variable_mappings[source])
    df = df[var_names].sort_index()
    df_dict[source] = df
    print (f'source: {source}')
    print (df_dict[source])    